# Inflation: baseline

训练均值基线。按时间顺序使用前 90% 训练、后 10% 测试；此划分不一定等同于原表的划分。

In [ ]:
from pathlib import Path
import sys

# Locate the project from either the root or an experiments subfolder.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "requirements.txt").is_file() and (p / "src").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Please open this notebook inside time-series-prediction.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing.cleaning_pipeline import clean_time_series

series = clean_time_series(str(ROOT / "data" / "inflation.csv"), date_col="ds", value_col="y", missing_method="drop")["target"]
series.head()

In [ ]:
import numpy as np
import pandas as pd
from src.models.baseline import mean_forecast
cut = int(len(series) * 0.9)
train, test = series.iloc[:cut], series.iloc[cut:]
pred = mean_forecast(train.to_numpy(), len(test))
metrics = pd.DataFrame([{"model": "training_mean", "mse": np.mean((test.to_numpy()-pred)**2)}])
metrics.to_csv(ROOT / "results/inflation/metrics/baseline.csv", index=False)
metrics